# 04 — Inventory Optimization

Use PuLP to solve the multi-item inventory optimization problem.
Minimize holding and stockout costs subject to capacity and budget constraints.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from src.optimize import InventoryInput, solve_inventory_optimization
from src.config import INVENTORY_PARAMS

print('Libraries loaded.')

## Define Sample Inventory Data

In [ ]:
items = [
    InventoryInput('bed_bath_table', 120, 20, 50, 45, 75, 7, 1.0),
    InventoryInput('health_beauty', 85, 15, 40, 55, 90, 7, 0.8),
    InventoryInput('sports_leisure', 60, 12, 30, 65, 100, 7, 1.2),
    InventoryInput('furniture_decor', 40, 10, 25, 80, 130, 14, 2.0),
    InventoryInput('computers_accessories', 95, 18, 60, 200, 350, 5, 0.5),
    InventoryInput('toys', 50, 12, 35, 30, 55, 7, 0.6),
    InventoryInput('garden_tools', 30, 8, 20, 70, 110, 10, 1.5),
    InventoryInput('books', 70, 15, 45, 25, 45, 5, 0.3),
]

print(f'{len(items)} categories defined')
print('\nInput summary:')
for item in items:
    print(f'  {item.category:25s} demand={item.forecast_demand:4.0f}  '
          f'stock={item.current_stock:4.0f}  cost=${item.unit_cost:6.2f}  '
          f'lead={item.lead_time_days}d')

## Run Optimization

In [ ]:
result = solve_inventory_optimization(
    items,
    storage_capacity=5000,
    budget=300000,
    service_level=0.95,
)

print(f'Status: {result["status"]}')
print(f'Total Cost: ${result["total_cost"]:,.2f}')

In [ ]:
results_df = result['results']
results_df

## Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Reorder quantities
cats = results_df['category']
axes[0].barh(range(len(cats)), results_df['reorder_quantity'].values[::-1], color='steelblue')
axes[0].set_yticks(range(len(cats)))
axes[0].set_yticklabels(cats[::-1])
axes[0].set_title('Optimal Reorder Quantities')
axes[0].set_xlabel('Units')

# Safety stock
axes[1].bar(range(len(cats)), results_df['reorder_point'], alpha=0.7, label='Reorder Point')
axes[1].bar(range(len(cats)), results_df['safety_stock'], alpha=0.7, label='Safety Stock')
axes[1].set_xticks(range(len(cats)))
axes[1].set_xticklabels(cats, rotation=45, ha='right')
axes[1].set_title('Reorder Point & Safety Stock')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Current vs recommended
fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(len(cats))
w = 0.35

ax.bar(x - w/2, results_df['current_stock'], w, label='Current Stock', color='gray')
ax.bar(x + w/2, results_df['total_inventory_after_order'], w, label='After Reorder', color='#2E86AB')

ax.set_xticks(x)
ax.set_xticklabels(cats, rotation=45, ha='right')
ax.set_ylabel('Units')
ax.set_title('Current vs Recommended Inventory Levels')
ax.legend()
plt.tight_layout()
plt.show()

## Cost Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(cats, results_df['holding_cost'], color='coral')
ax.set_xlabel('Category')
ax.set_ylabel('Holding Cost ($)')
ax.set_title('Holding Cost by Category')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print(f'Total holding cost: ${results_df["holding_cost"].sum():,.2f}')
print(f'Total units to order: {results_df["reorder_quantity"].sum():,.0f}')

## Sensitivity Analysis

In [ ]:
# Test different service levels
service_levels = [0.85, 0.90, 0.95, 0.99]
total_costs = []

for sl in service_levels:
    r = solve_inventory_optimization(items, service_level=sl)
    total_costs.append(r['total_cost'])

fig, ax = plt.subplots()
ax.plot(service_levels, total_costs, marker='o', linewidth=2, color='#2E86AB')
ax.set_xlabel('Service Level')
ax.set_ylabel('Total Cost ($)')
ax.set_title('Service Level vs Total Cost')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

for sl, cost in zip(service_levels, total_costs):
    print(f'Service Level {sl:.0%}: Total Cost = ${cost:,.2f}')